## Prerequisite Code

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/sd-warrior-acquisition/initial-setup/03-utils

In [0]:
# Verify loaded utils
print(wr_bronze_schema)
print(wr_silver_schema)
print(wr_gold_schema)

print(sd_gold_schema)

warrior_bronze
warrior_silver
warrior_gold
sportsdirect_gold


In [0]:
# Define Notebook Widgets
dbutils.widgets.text('catalog', 'sportsdirect_sales', 'Catalog')
dbutils.widgets.text('data_source', 'customers', 'Data Source')

# Access widget values
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

# Define the source directory
source_dir = f's3://sd-warrior-acquisition/{data_source}/*.csv'

In [0]:
# View values
print(catalog)
print(data_source)

print(source_dir)

sportsdirect_sales
customers
s3://sd-warrior-acquisition/customers/*.csv


## Warrior Bronze Layer

In [0]:
# Get raw customer data into a dataframe
raw_data = spark.read \
            .format('csv') \
            .option('header', True) \
            .option('inferSchema', True) \
            .load(source_dir) \
            .withColumn('read_timestamp', F.current_timestamp()) \
            .select('*', '_metadata.file_name', '_metadata.file_size')

In [0]:
# View schema
raw_data.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
# View data
display(raw_data.limit(10))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-03-16T13:12:36.670Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-03-16T13:12:36.670Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-03-16T13:12:36.670Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-03-16T13:12:36.670Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-03-16T13:12:36.670Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-03-16T13:12:36.670Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-03-16T13:12:36.670Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-03-16T13:12:36.670Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-03-16T13:12:36.670Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-03-16T13:12:36.670Z,customers.csv,1404


In [0]:
# Write raw data to bronze table
raw_data.write \
    .format('delta') \
    .option('delta.enableChangeDataFeed', True) \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{wr_bronze_schema}.dim_customer')

In [0]:
# View data
display(spark.sql(f'SELECT * FROM {catalog}.{wr_bronze_schema}.dim_customer'))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-03-16T13:12:45.616Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-03-16T13:12:45.616Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-03-16T13:12:45.616Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-03-16T13:12:45.616Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-03-16T13:12:45.616Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-03-16T13:12:45.616Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-03-16T13:12:45.616Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-03-16T13:12:45.616Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-03-16T13:12:45.616Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-03-16T13:12:45.616Z,customers.csv,1404


## Warrior Silver Layer

In [0]:
# Apply transformations on raw data

# Drop duplicate records

# View duplicates
display(raw_data.groupBy('customer_id').count().filter(F.col('count') > 1))

# Drop duplicates
transformed_data = raw_data.dropDuplicates(['customer_id'])

customer_id,count
789321,2
789503,2
789522,2
789603,2


In [0]:
# Display row count of transformed data
display(transformed_data.count())

35

In [0]:
# Trim spaces in customer names

# View impacted records
display(transformed_data.filter(F.col('customer_name') != F.trim(F.col('customer_name'))))

# Trim spaces
transformed_data = transformed_data.withColumn('customer_name', F.trim(F.col('customer_name')))

# Check for records with spaces
display(transformed_data.filter(F.col('customer_name') != F.trim(F.col('customer_name'))))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789121,HydroBoost Nutrition,Hyderabad,2026-03-16T13:12:52.591Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-03-16T13:12:52.591Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-03-16T13:12:52.591Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-03-16T13:12:52.591Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-03-16T13:12:52.591Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-03-16T13:12:52.591Z,customers.csv,1404


customer_id,customer_name,city,read_timestamp,file_name,file_size


In [0]:
# Fix casing

# View impacted records
display(transformed_data.select('customer_name').distinct())

# Fix casing
transformed_data = transformed_data.withColumn(
    'customer_name',
    F.initcap(F.col('customer_name'))
)

# Verify transformation
display(transformed_data.select('customer_name').distinct())

customer_name
EliteAthlete Nutrition
PowerSnack hub
Recovery Lane
GamePlan Foods
FitFuel Market
Athlete's Choice Store
ZenAthlete foods
HydroBoost Nutrition
ZenAthlete Foods
champion's Choice


customer_name
Eliteathlete Nutrition
Powersnack Hub
Recovery Lane
Gameplan Foods
Fitfuel Market
Athlete's Choice Store
Zenathlete Foods
Hydroboost Nutrition
Champion's Choice
Peak Performance Store


In [0]:
# Fix typos

# View impacted records
transformed_data.select('city').distinct().show()

# Fix typos
transformed_data = transformed_data.withColumn(
    'city',
    F.when(F.col('city').isin(['Hyderbad', 'Hyderabadd']), 'Hyderabad')
    .when(F.col('city').isin(['Bengaluruu', 'Bengalore']), 'Bengaluru')
    .when(F.col('city').isin(['NewDelhi', 'NewDheli', 'NewDelhee']), 'New Delhi')
    .otherwise(F.col('city'))
)

# Verify transformation
transformed_data.select('city').distinct().show()

+----------+
|      city|
+----------+
| New Delhi|
| Hyderabad|
| Bengaluru|
|      NULL|
|  Hyderbad|
|  NewDelhi|
|Bengaluruu|
| Bengalore|
|  NewDheli|
|Hyderabadd|
| NewDelhee|
+----------+

+---------+
|     city|
+---------+
|New Delhi|
|Hyderabad|
|Bengaluru|
|     NULL|
+---------+



In [0]:
display(transformed_data)

customer_id,customer_name,city,read_timestamp,file_name,file_size
789101,Endurance Foods,Bengaluru,2026-03-16T13:13:00.214Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-03-16T13:13:00.214Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-03-16T13:13:00.214Z,customers.csv,1404
789121,Hydroboost Nutrition,Hyderabad,2026-03-16T13:13:00.214Z,customers.csv,1404
789122,Hydroboost Nutrition,New Delhi,2026-03-16T13:13:00.214Z,customers.csv,1404
789201,Fitfuel Market,Bengaluru,2026-03-16T13:13:00.214Z,customers.csv,1404
789202,Fitfuel Market,Hyderabad,2026-03-16T13:13:00.214Z,customers.csv,1404
789203,Fitfuel Market,New Delhi,2026-03-16T13:13:00.214Z,customers.csv,1404
789220,Macrobite Superfoods,Bengaluru,2026-03-16T13:13:00.214Z,customers.csv,1404
789221,Macrobite Superfoods,Hyderabad,2026-03-16T13:13:00.214Z,customers.csv,1404


In [0]:
# Fill empty cells

# View empty cell records
display(transformed_data.select('*').filter(F.col('city').isNull()))

# Create empty cells customer list
empty_cells_customer = [row['customer_name'] for row in transformed_data.select('customer_name').filter(F.col('city').isNull()).collect()]

# View empty cells customer list
print(empty_cells_customer)

# Display records with empty cell customers
display(transformed_data.filter(F.col('customer_name').isin(empty_cells_customer)))

# Fill empty cells

customer_city = {
    789403: 'New Delhi',
    789420: 'Bengaluru',
    789521: 'Hyderabad',
    789603: 'Hyderabad'
}

fix_data = spark.createDataFrame(customer_city.items(), ['customer_id', 'fix_city'])

transformed_data = transformed_data.join(
    fix_data,
    on='customer_id',
    how='left'
) \
.withColumn(
    'city',
    F.coalesce('city', 'fix_city')
) \
.drop('fix_city')

# View transformed data
display(transformed_data)

customer_id,customer_name,city,read_timestamp,file_name,file_size
789403,Sprintx Nutrition,null,2026-03-16T13:42:25.538Z,customers.csv,1404
789420,Zenathlete Foods,null,2026-03-16T13:42:25.538Z,customers.csv,1404
789521,Primefuel Nutrition,null,2026-03-16T13:42:25.538Z,customers.csv,1404
789603,Recovery Lane,null,2026-03-16T13:42:25.538Z,customers.csv,1404


['Zenathlete Foods', 'Sprintx Nutrition', 'Primefuel Nutrition', 'Recovery Lane']


customer_id,customer_name,city,read_timestamp,file_name,file_size
789401,Sprintx Nutrition,Bengaluru,2026-03-16T13:42:28.792Z,customers.csv,1404
789402,Sprintx Nutrition,Hyderabad,2026-03-16T13:42:28.792Z,customers.csv,1404
789403,Sprintx Nutrition,null,2026-03-16T13:42:28.792Z,customers.csv,1404
789420,Zenathlete Foods,null,2026-03-16T13:42:28.792Z,customers.csv,1404
789421,Zenathlete Foods,Hyderabad,2026-03-16T13:42:28.792Z,customers.csv,1404
789422,Zenathlete Foods,New Delhi,2026-03-16T13:42:28.792Z,customers.csv,1404
789520,Primefuel Nutrition,Bengaluru,2026-03-16T13:42:28.792Z,customers.csv,1404
789521,Primefuel Nutrition,null,2026-03-16T13:42:28.792Z,customers.csv,1404
789522,Primefuel Nutrition,New Delhi,2026-03-16T13:42:28.792Z,customers.csv,1404
789601,Recovery Lane,Bengaluru,2026-03-16T13:42:28.792Z,customers.csv,1404


customer_id,customer_name,city,read_timestamp,file_name,file_size
789101,Endurance Foods,Bengaluru,2026-03-16T13:42:30.194Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-03-16T13:42:30.194Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-03-16T13:42:30.194Z,customers.csv,1404
789121,Hydroboost Nutrition,Hyderabad,2026-03-16T13:42:30.194Z,customers.csv,1404
789122,Hydroboost Nutrition,New Delhi,2026-03-16T13:42:30.194Z,customers.csv,1404
789201,Fitfuel Market,Bengaluru,2026-03-16T13:42:30.194Z,customers.csv,1404
789202,Fitfuel Market,Hyderabad,2026-03-16T13:42:30.194Z,customers.csv,1404
789203,Fitfuel Market,New Delhi,2026-03-16T13:42:30.194Z,customers.csv,1404
789220,Macrobite Superfoods,Bengaluru,2026-03-16T13:42:30.194Z,customers.csv,1404
789221,Macrobite Superfoods,Hyderabad,2026-03-16T13:42:30.194Z,customers.csv,1404


In [0]:
# Verify transformation
display(transformed_data.filter(F.col('city').isNull()).count())

0

In [0]:
# Convert data type to string

# Check customer_id data type
transformed_data.printSchema() # integer

# Convert data type
transformed_data = transformed_data.withColumn(
    'customer_id',
    F.col('customer_id').cast('string')
)

# Verify transformation
transformed_data.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
# Match Warrior customer data structure with Sports Direct customer data structure

transformed_data = transformed_data.withColumn(
    'customer',
    F.concat(F.col('customer_name'), F.lit('-'), F.col('city'))
) \
.withColumn('market', F.lit('India')) \
.withColumn('platform', F.lit('Sports Bar')) \
.withColumn('channel', F.lit('Acquisition'))

# Rename column
transformed_data = transformed_data.withColumnRenamed('customer_id', 'customer_code')

# View transformed data
display(transformed_data)

customer_code,customer_name,city,read_timestamp,file_name,file_size,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,2026-03-16T14:12:18.574Z,customers.csv,1404,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,2026-03-16T14:12:18.574Z,customers.csv,1404,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-03-16T14:12:18.574Z,customers.csv,1404,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,2026-03-16T14:12:18.574Z,customers.csv,1404,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,2026-03-16T14:12:18.574Z,customers.csv,1404,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store,Bengaluru,2026-03-16T14:12:18.574Z,customers.csv,1404,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods,Bengaluru,2026-03-16T14:12:18.574Z,customers.csv,1404,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market,Hyderabad,2026-03-16T14:12:18.574Z,customers.csv,1404,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition,New Delhi,2026-03-16T14:12:18.574Z,customers.csv,1404,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods,Hyderabad,2026-03-16T14:12:18.574Z,customers.csv,1404,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


In [0]:
# Write transformed data to the silver table
transformed_data.write \
    .format('delta') \
    .option('enableChangeDataFeed', 'true') \
    .option('mergeSchema', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{wr_silver_schema}.dim_customer')

## Warrior Gold Layer

In [0]:
# Get analytics data from the silver layer
analytics_data = transformed_data.select('customer_code', 'customer', 'market', 'platform', 'channel')

# View analytics data
display(analytics_data)

# Write analytics data to the gold table
analytics_data.write \
    .format('delta') \
    .option('enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{wr_gold_schema}.dim_customer')

customer_code,customer,market,platform,channel
789622,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition
789301,Athlete's Choice Store-Bengaluru,India,Sports Bar,Acquisition
789420,Zenathlete Foods-Bengaluru,India,Sports Bar,Acquisition
789202,Fitfuel Market-Hyderabad,India,Sports Bar,Acquisition
789122,Hydroboost Nutrition-New Delhi,India,Sports Bar,Acquisition
789421,Zenathlete Foods-Hyderabad,India,Sports Bar,Acquisition


## Sports Direct Gold Layer

In [0]:
# Merge Warrior gold layer data with Sports Direct gold layer data

sd_dim_customer_table = DeltaTable.forName(spark, 'sportsdirect_sales.sportsdirect_gold.dim_customer')

# Count records before merge
display(spark.table('sportsdirect_sales.sportsdirect_gold.dim_customer').count())

#  Merge data
sd_dim_customer_table.alias('target').merge(
    source=analytics_data.alias('source'),
    condition='target.customer_code = source.customer_code'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Count records after merge
display(spark.table('sportsdirect_sales.sportsdirect_gold.dim_customer').count())

18

53